# WTI Producer Hedge Simulator

I built this notebook to see how much a crude producer could reduce revenue risk by hedging expected production with WTI futures.

The idea is simple: if oil prices fall, physical revenue drops. A short futures hedge can help offset part of that move.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Load the WTI data

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare a few hedge sizes

Here I compare different percentages of the same 100,000 barrels of monthly production.

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## What if crude falls 25%?

This is a simple stress test using a 75% hedge.

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Midland vs. Cushing basis risk

A WTI futures hedge can reduce the main oil-price risk, but Midland and Cushing can still move differently.

That leftover difference is basis risk. The examples below are simple stress tests.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### What this means

A full WTI hedge does not remove every risk. If Midland gets cheaper compared with Cushing, the producer can still take a hit. A separate basis hedge can help with that.

## Which hedge size worked best?

Here I use the historical spot and futures relationship to estimate the hedge size that reduced price swings the most.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
